In [ ]:
from pathlib import Path

# ===============================
# 路径设置（沿用你原来的逻辑）
# ===============================
working_dir = Path.cwd().resolve()
root_dir = next((p for p in [working_dir, *working_dir.parents] if (p / "data").is_dir() and (p / "model").is_dir()), None)
if root_dir is None:
    raise RuntimeError("Run this notebook from within the repository.")
model_dir = root_dir / "model"
poisson_dir = model_dir / "poisson"
input_dir = root_dir / "data" / "processed" / "goal-filtered"
# 建议单独一个输出目录，避免和 MLE 混在一起
output_dir = poisson_dir / "lambdas" / "bayes-hierarchical-lambdas"



# Posterior predictive 的输出目录
ppc_output_dir = poisson_dir / "posterior_predictive"
ppc_output_dir.mkdir(parents=True, exist_ok=True)

print("input_dir:", input_dir)
print("ppc_output_dir:", ppc_output_dir)


In [ ]:
import numpy as np
import pandas as pd
import arviz as az

# --------------------------------------------------------------
# Posterior Predictive Simulation for a single season
# --------------------------------------------------------------

def posterior_predictive_simulation(
    year: str,
    trace,
    num_sim_seasons: int = 1000
):
    """
    根据 posterior samples（trace）对整个赛季进行 Posterior Predictive Simulation。

    输入：
        year: 赛季年份字符串（如 "1998-1999"）
        trace: PyMC InferenceData（从 .nc 文件加载）
        num_sim_seasons: 模拟多少次整个赛季（默认 1000）

    输出：
        summary_df: 每支球队的积分统计结果
        sim_results: 模拟包含 (num_sim_seasons × num_teams) 的排名积分矩阵
    """

    # ----------------------------------------------------------
    # 读取真实赛季比赛数据（必须与建模阶段格式一致）
    # ----------------------------------------------------------
    input_file = input_dir / f"{year}.csv"
    data = pd.read_csv(input_file)

    teams = sorted(set(data["HomeTeam"]).union(data["AwayTeam"]))
    num_teams = len(teams)
    team_to_idx = {team: i for i, team in enumerate(teams)}

    home_idx = data['HomeTeam'].map(team_to_idx).values
    away_idx = data['AwayTeam'].map(team_to_idx).values

    num_matches = len(data)

    # ----------------------------------------------------------
    # 提取 posterior samples
    # alpha_att: shape (chain, draw, num_teams)
    # ----------------------------------------------------------
    alpha_att = trace.posterior["alpha_att"].stack(sample=("chain", "draw")).values
    alpha_def = trace.posterior["alpha_def"].stack(sample=("chain", "draw")).values
    C_samples   = trace.posterior["C"].stack(sample=("chain", "draw")).values

    num_samples = alpha_att.shape[1]

    print(f"Season {year} has {num_teams} teams, {num_matches} matches.")
    print(f"Posterior samples loaded: {num_samples} MCMC draws.")
    print(f"Simulating {num_sim_seasons} seasons...\n")

    # ----------------------------------------------------------
    # 存储每次模拟的积分结果
    # shape = (num_sim_seasons, num_teams)
    # ----------------------------------------------------------
    sim_points = np.zeros((num_sim_seasons, num_teams), dtype=int)

    rng = np.random.default_rng(42)

    # ----------------------------------------------------------
    # 主循环：模拟 num_sim_seasons 轮完整赛季
    # ----------------------------------------------------------
    for s in range(num_sim_seasons):

        # 本赛季积分，初始化为 0
        points = np.zeros(num_teams, dtype=int)

        # 随机抽取一个 posterior 样本 index
        idx = rng.integers(0, num_samples)

        att_sample = alpha_att[:, idx]   # shape = (num_teams,)
        def_sample = alpha_def[:, idx]
        C_sample   = C_samples[idx]      # scalar

        # --------- 逐场比赛模拟 ---------
        for m in range(num_matches):
            h = home_idx[m]
            a = away_idx[m]

            # Poisson 强度：exp(attack_home - defense_away + C)
            home_lambda = np.exp(att_sample[h] - def_sample[a] + C_sample)
            away_lambda = np.exp(att_sample[a] - def_sample[h])

            # 抽样进球数
            hg = rng.poisson(home_lambda)
            ag = rng.poisson(away_lambda)

            # 记积分
            if hg > ag:
                points[h] += 3
            elif ag > hg:
                points[a] += 3
            else:
                points[h] += 1
                points[a] += 1

        sim_points[s, :] = points

    # ----------------------------------------------------------
    # 输出统计内容
    # ----------------------------------------------------------

    summary_df = pd.DataFrame({
        "Team": teams,
        "Points_mean": sim_points.mean(axis=0),
        "Points_median": np.median(sim_points, axis=0),
        "Points_sd": sim_points.std(axis=0),
        "Points_2.5%": np.percentile(sim_points, 2.5, axis=0),
        "Points_97.5%": np.percentile(sim_points, 97.5, axis=0),
    })

    # 冠军概率（第一名概率）
    champions = np.argmax(sim_points, axis=1)
    for i, team in enumerate(teams):
        summary_df.loc[i, "Champion_prob"] = np.mean(champions == i)

    # 前四概率
    for i, team in enumerate(teams):
        summary_df.loc[i, "Top4_prob"] = np.mean(
            np.argsort(sim_points, axis=1)[:, -4:] == i
        )

    # 降级概率（Bottom 3）
    for i, team in enumerate(teams):
        summary_df.loc[i, "Relegation_prob"] = np.mean(
            np.argsort(sim_points, axis=1)[:, :3] == i
        )

    return summary_df, sim_points


In [ ]:
year = "1993-1994"
trace = az.from_netcdf(output_dir / f"{year}-trace.nc")
summary_df, sim_points = posterior_predictive_simulation(year, trace, num_sim_seasons=1000)
summary_df
